# Lab 05-02: Data Licensing and Compliance**Module:** 05 — Governance (8% of exam)  **UI Sections:** Catalog | Marketplace | Discover  **Estimated Time:** 30–40 minutes  **Difficulty:** Beginner–Intermediate---## Learning ObjectivesBy the end of this lab you will be able to:1. Distinguish the major open-source license families (Apache 2.0, MIT, GPL, Llama Community)2. Identify where license information lives in Databricks (Marketplace model cards, Discover page)3. Explain when fine-tuning on licensed data creates a "derivative work"4. Build a practical license compliance checklist for a GenAI project5. Use Unity Catalog lineage to trace data provenance for license audits## Exam Objectives Covered| Objective | Tested Here ||-----------|-------------|| Understand data licensing implications for GenAI | Steps 1–3 || Evaluate model licenses before deployment | Steps 2, 4 || Trace data lineage for compliance | Step 5 |

---## What and WhyEvery model you deploy and every dataset you feed into a GenAI application comes with **legal stringsattached**. Ignoring licenses can lead to lawsuits, forced open-sourcing of proprietary code, ortakedown orders — none of which look good on a quarterly report.The exam won't ask you to recite license text, but it **will** test whether you understand:- **Which licenses allow commercial use** and which don't- **What "derivative work" means** when you fine-tune on licensed data- **Where to find license info** in the Databricks UI- **How Unity Catalog lineage** helps with compliance audits---

> **Mental Model: Recipe Rights**>> Think of data and model licenses like recipe rights:>> | License Type | Recipe Analogy |> |-------------|----------------|> | **Apache 2.0 / MIT** | "Use my recipe freely. Sell dishes in your restaurant. Just credit me." |> | **GPL (Copyleft)** | "Use my recipe, but if you modify it, you MUST publish YOUR version too." |> | **Llama Community** | "Use my recipe freely up to 700M servings/month. Credit me. Don't use it to build a competing restaurant chain." |> | **Proprietary** | "Pay me per dish. Don't share my recipe. Don't reverse-engineer it." |>> **Key insight:** The license determines what you can *do* with the output, not just> whether you can *read* the source.---

## Exam Alert| Trap | Correct Answer ||------|---------------|| "All open-source models can be used commercially" | Check the license — some restrict commercial use (e.g., certain research-only models) || "Fine-tuning on licensed data is always allowed" | The **training data** license determines what you can do with derivative works || "Marketplace models are license-free" | Each Marketplace model has its own license — **always check** before deploying || "MIT and Apache 2.0 are the same" | Both are permissive, but Apache 2.0 includes an explicit **patent grant** (more protection) || "Open-source means no restrictions" | Open-source means source is available — restrictions vary widely by license |---

## Databricks UI Navigation**Finding license info for a Marketplace model:****UI →** Left nav → **Discover** → Search for a model → Click model card → **License** section**Finding license info for a Foundation Model:****UI →** Left nav → **Serving** → Select endpoint → Model details link → License info**Checking data lineage:****UI →** Left nav → **Catalog** → Navigate to table → **Lineage** tab---

## SetupThis lab is primarily conceptual — no heavy libraries needed. We'll use Python to buildinteractive exercises and a compliance checklist tool.

In [ ]:
import json
from datetime import datetime

# Configuration
CATALOG = "main"
SCHEMA = "genai_labs"

print("Lab 05-02: Data Licensing and Compliance")
print(f"Date: {datetime.now().strftime('%Y-%m-%d')}")
print("Ready to explore licensing!")

**Expected output:**```Lab 05-02: Data Licensing and ComplianceDate: 2026-03-18Ready to explore licensing!```---

## Step 1: The Five License FamiliesEvery model and dataset you encounter in the Databricks ecosystem falls into one of thesefive categories. Let's build a reference table and test your understanding.> **Analogy:** Licenses are like parking signs. Some say "Park free anytime" (MIT),> some say "Park free but leave your keys for others" (GPL), and some say> "Paid parking only, tow zone after hours" (Proprietary). Reading the sign> before you park saves you from an expensive ticket.

In [ ]:
# License reference database
LICENSES = {
    "Apache 2.0": {
        "type": "Permissive",
        "commercial_use": True,
        "modification": True,
        "distribution": True,
        "patent_grant": True,
        "copyleft": False,
        "examples": ["DBRX", "Most Apache Spark code", "Many Hugging Face models"],
        "key_requirement": "Include license notice and state changes",
        "exam_note": "PREFERRED for enterprise — patent grant protects you"
    },
    "MIT": {
        "type": "Permissive",
        "commercial_use": True,
        "modification": True,
        "distribution": True,
        "patent_grant": False,
        "copyleft": False,
        "examples": ["Many npm packages", "Some smaller ML models"],
        "key_requirement": "Include copyright notice",
        "exam_note": "Simple and permissive, but NO patent protection"
    },
    "GPL v3": {
        "type": "Copyleft",
        "commercial_use": True,
        "modification": True,
        "distribution": True,
        "patent_grant": True,
        "copyleft": True,
        "examples": ["Linux kernel", "Some research models"],
        "key_requirement": "Derivative works MUST also be GPL",
        "exam_note": "DANGER — if you modify and distribute, YOUR code becomes GPL too"
    },
    "Llama Community": {
        "type": "Semi-permissive",
        "commercial_use": True,
        "modification": True,
        "distribution": True,
        "patent_grant": False,
        "copyleft": False,
        "examples": ["Llama 2", "Llama 3", "Code Llama"],
        "key_requirement": "Credit Meta; >700M monthly users need special license",
        "exam_note": "Free for most companies — check the 700M user threshold"
    },
    "Proprietary": {
        "type": "Restricted",
        "commercial_use": "Depends on agreement",
        "modification": False,
        "distribution": False,
        "patent_grant": False,
        "copyleft": False,
        "examples": ["GPT-4 (via API)", "Some enterprise datasets"],
        "key_requirement": "Pay license fee; follow usage terms exactly",
        "exam_note": "Used via API only — you never get the weights"
    }
}

# Print comparison table
print(f"{'License':<18} {'Type':<15} {'Commercial':<12} {'Modify':<10} {'Copyleft':<10} {'Patent':<8}")
print("-" * 73)
for name, info in LICENSES.items():
    commercial = "Yes" if info["commercial_use"] is True else "Varies"
    modify = "Yes" if info["modification"] else "No"
    copyleft = "YES!" if info["copyleft"] else "No"
    patent = "Yes" if info["patent_grant"] else "No"
    print(f"{name:<18} {info['type']:<15} {commercial:<12} {modify:<10} {copyleft:<10} {patent:<8}")

print()
print("KEY EXAM INSIGHT: Apache 2.0 is preferred for enterprise because")
print("it includes a patent grant that MIT lacks.")

**Expected output:**```License            Type            Commercial   Modify     Copyleft   Patent  -------------------------------------------------------------------------Apache 2.0         Permissive      Yes          Yes        No         Yes     MIT                Permissive      Yes          Yes        No         No      GPL v3             Copyleft        Yes          Yes        YES!       Yes     Llama Community    Semi-permissive Yes          Yes        No         No      Proprietary        Restricted      Varies       No         No         No      KEY EXAM INSIGHT: Apache 2.0 is preferred for enterprise becauseit includes a patent grant that MIT lacks.```---

## Step 2: Model Licensing on DatabricksDatabricks surfaces license information in several places. Knowing **where to look** isan exam skill.There are three places you'll encounter models on Databricks, each with differentlicensing implications:| Source | License Location | What to Check ||--------|-----------------|---------------|| **Foundation Model APIs** | Pre-vetted by Databricks | Pay-per-token; license handled for you || **Marketplace Models** | Model card page | Click the model → read the License section || **External Models (AI Gateway)** | Provider's terms | OpenAI TOS, Anthropic Usage Policy, etc. |

In [ ]:
# Simulated model registry with license info
DATABRICKS_MODELS = {
    "databricks-meta-llama-3-3-70b-instruct": {
        "source": "Foundation Model API",
        "license": "Llama Community",
        "commercial_ok": True,
        "fine_tune_ok": True,
        "special_terms": "Attribution required; >700M MAU need Meta approval",
        "where_to_check": "UI -> Serving -> Endpoint details"
    },
    "databricks-dbrx-instruct": {
        "source": "Foundation Model API",
        "license": "Apache 2.0",
        "commercial_ok": True,
        "fine_tune_ok": True,
        "special_terms": "None — fully open",
        "where_to_check": "UI -> Serving -> Endpoint details"
    },
    "databricks-bge-large-en": {
        "source": "Foundation Model API",
        "license": "MIT",
        "commercial_ok": True,
        "fine_tune_ok": True,
        "special_terms": "None — permissive",
        "where_to_check": "UI -> Serving -> Endpoint details"
    },
    "marketplace/medical-ner-model": {
        "source": "Marketplace",
        "license": "Research Only",
        "commercial_ok": False,
        "fine_tune_ok": False,
        "special_terms": "Academic use only — NO commercial deployment",
        "where_to_check": "UI -> Discover -> Model card -> License"
    },
    "external/gpt-4": {
        "source": "AI Gateway (External)",
        "license": "Proprietary (OpenAI TOS)",
        "commercial_ok": True,
        "fine_tune_ok": "Via OpenAI API only",
        "special_terms": "Usage limits; content policy; no weight access",
        "where_to_check": "OpenAI Terms of Service"
    }
}

print("MODEL LICENSE AUDIT")
print("=" * 80)
for model, info in DATABRICKS_MODELS.items():
    status = "APPROVED" if info["commercial_ok"] is True else "BLOCKED"
    marker = "+" if status == "APPROVED" else "X"
    print(f"  [{marker}] {model}")
    print(f"      Source: {info['source']}  |  License: {info['license']}")
    print(f"      Commercial: {info['commercial_ok']}  |  Fine-tune: {info['fine_tune_ok']}")
    if info['special_terms'] != "None — fully open":
        print(f"      ** {info['special_terms']}")
    print()

print("EXAM TIP: Foundation Model APIs are pre-vetted — license is handled.")
print("Marketplace models require YOU to check the license on the model card.")

**Expected output:**```MODEL LICENSE AUDIT================================================================================  [+] databricks-meta-llama-3-3-70b-instruct      Source: Foundation Model API  |  License: Llama Community      Commercial: True  |  Fine-tune: True      ** Attribution required; >700M MAU need Meta approval  [+] databricks-dbrx-instruct      Source: Foundation Model API  |  License: Apache 2.0      Commercial: True  |  Fine-tune: True  ...  [X] marketplace/medical-ner-model      Source: Marketplace  |  License: Research Only      Commercial: False  |  Fine-tune: False      ** Academic use only — NO commercial deployment```---

## Step 3: Data Licensing and Derivative WorksThis is where licensing gets tricky for GenAI. When you **fine-tune** a model on a dataset,the resulting model may be a **derivative work** of that dataset — subject to the dataset'slicense terms.> **Analogy:** If you write a cookbook (fine-tuned model) that copies recipes (training data)> from another book, the original book's copyright applies to your cookbook.> But if you just *read* the book for inspiration and write original recipes,> it's usually fine. Fine-tuning is more like copying than inspiration.### The Derivative Works Test| Action | Derivative Work? | License Applies? ||--------|-----------------|-----------------|| Fine-tune on the data | **Yes** — model weights encode the data | **Yes** || RAG retrieval (data stays in place) | **No** — data is referenced, not absorbed | **Usually no** (but check data access terms) || Prompt engineering with examples | **No** — examples are inputs, not training | **No** || Distillation from a licensed model | **Yes** — student model derives from teacher | **Yes** |

In [ ]:
# Interactive scenario quiz: Is this a derivative work?
scenarios = [
    {
        "scenario": "You fine-tune Llama 3 on your company's internal Q&A data",
        "derivative_of_model": True,
        "derivative_of_data": False,
        "explanation": "Derivative of Llama (Llama license applies). Your internal data is yours.",
    },
    {
        "scenario": "You fine-tune an MIT-licensed model on GPL-licensed research papers",
        "derivative_of_model": True,
        "derivative_of_data": True,
        "explanation": "GPL is copyleft — your fine-tuned model MUST be released under GPL!",
    },
    {
        "scenario": "You use RAG to retrieve from a proprietary database at query time",
        "derivative_of_model": False,
        "derivative_of_data": False,
        "explanation": "RAG doesn't create derivatives — data stays in place. But check data ACCESS terms.",
    },
    {
        "scenario": "You distill GPT-4 outputs into a smaller model",
        "derivative_of_model": True,
        "derivative_of_data": False,
        "explanation": "OpenAI TOS may prohibit this! Always check provider terms on distillation.",
    },
    {
        "scenario": "You use few-shot examples from a CC-BY dataset in your prompts",
        "derivative_of_model": False,
        "derivative_of_data": False,
        "explanation": "Prompt examples are runtime inputs, not training. CC-BY just needs attribution.",
    },
]

print("DERIVATIVE WORKS QUIZ")
print("=" * 70)
for i, s in enumerate(scenarios, 1):
    print(f"\nScenario {i}: {s['scenario']}")
    model_status = "YES" if s["derivative_of_model"] else "no"
    data_status = "YES" if s["derivative_of_data"] else "no"
    print(f"  Derivative of model? {model_status}")
    print(f"  Derivative of data?  {data_status}")
    print(f"  --> {s['explanation']}")

print("\n" + "=" * 70)
print("EXAM KEY: Fine-tuning creates derivatives. RAG does NOT.")
print("This distinction is heavily tested!")

**Expected output:**```DERIVATIVE WORKS QUIZ======================================================================Scenario 1: You fine-tune Llama 3 on your company's internal Q&A data  Derivative of model? YES  Derivative of data?  no  --> Derivative of Llama (Llama license applies). Your internal data is yours.Scenario 2: You fine-tune an MIT-licensed model on GPL-licensed research papers  Derivative of model? YES  Derivative of data?  YES  --> GPL is copyleft — your fine-tuned model MUST be released under GPL!...```---

## Step 4: License Compliance ChecklistIn practice, every GenAI project should go through a compliance review. Let's builda reusable checklist tool that you can use in real projects.This exercise mirrors what a governance team would do before deploying a GenAIapplication to production.

In [ ]:
def license_compliance_check(project_config):
    """Run a compliance check on a GenAI project configuration."""
    issues = []
    warnings = []
    approved = []

    # Check base model license
    model = project_config.get("base_model", {})
    if model.get("license") == "Research Only":
        issues.append(f"BLOCKER: {model['name']} is research-only — cannot deploy commercially")
    elif model.get("license") == "GPL v3":
        issues.append(f"BLOCKER: {model['name']} is GPL — your entire application must be open-sourced")
    elif model.get("license") in ["Apache 2.0", "MIT"]:
        approved.append(f"Model {model['name']} ({model['license']}) — commercial use OK")
    elif model.get("license") == "Llama Community":
        if project_config.get("monthly_active_users", 0) > 700_000_000:
            issues.append("BLOCKER: >700M MAU — need Meta's special license for Llama")
        else:
            approved.append(f"Model {model['name']} (Llama Community) — commercial use OK under 700M MAU")

    # Check training data
    for ds in project_config.get("training_data", []):
        if ds["license"] == "GPL v3":
            issues.append(f"BLOCKER: Training data '{ds['name']}' is GPL — fine-tuned model inherits GPL")
        elif ds["license"] == "Proprietary":
            warnings.append(f"WARNING: Check '{ds['name']}' license for derivative work clauses")
        else:
            approved.append(f"Training data '{ds['name']}' ({ds['license']}) — OK for fine-tuning")

    # Check RAG data sources
    for ds in project_config.get("rag_sources", []):
        if ds.get("access_agreement"):
            warnings.append(f"WARNING: RAG source '{ds['name']}' has access terms — review annually")
        else:
            approved.append(f"RAG source '{ds['name']}' — no derivative work risk (retrieval only)")

    # Check external API usage
    for api in project_config.get("external_apis", []):
        warnings.append(f"WARNING: External API '{api['name']}' — review TOS for output ownership")

    return {"issues": issues, "warnings": warnings, "approved": approved}


# Example project
my_project = {
    "project_name": "Customer Support RAG Bot",
    "monthly_active_users": 50_000,
    "base_model": {"name": "databricks-meta-llama-3-3-70b-instruct", "license": "Llama Community"},
    "training_data": [
        {"name": "Internal support tickets", "license": "Proprietary (internal)"},
        {"name": "Public FAQ dataset", "license": "Apache 2.0"},
    ],
    "rag_sources": [
        {"name": "Product documentation", "access_agreement": False},
        {"name": "Partner knowledge base", "access_agreement": True},
    ],
    "external_apis": [
        {"name": "OpenAI GPT-4 (fallback)", "provider": "OpenAI"}
    ]
}

result = license_compliance_check(my_project)

print(f"LICENSE COMPLIANCE REPORT: {my_project['project_name']}")
print("=" * 60)

if result["issues"]:
    print(f"\n BLOCKERS ({len(result['issues'])}):")
    for issue in result["issues"]:
        print(f"  {issue}")

if result["warnings"]:
    print(f"\n WARNINGS ({len(result['warnings'])}):")
    for warning in result["warnings"]:
        print(f"  {warning}")

if result["approved"]:
    print(f"\n APPROVED ({len(result['approved'])}):")
    for item in result["approved"]:
        print(f"  {item}")

status = "BLOCKED" if result["issues"] else ("REVIEW NEEDED" if result["warnings"] else "APPROVED")
print(f"\nOVERALL STATUS: {status}")

**Expected output:**```LICENSE COMPLIANCE REPORT: Customer Support RAG Bot============================================================ WARNINGS (3):  WARNING: Check 'Internal support tickets' license for derivative work clauses  WARNING: RAG source 'Partner knowledge base' has access terms — review annually  WARNING: External API 'OpenAI GPT-4 (fallback)' — review TOS for output ownership APPROVED (3):  Model databricks-meta-llama-3-3-70b-instruct (Llama Community) — commercial use OK under 700M MAU  Training data 'Public FAQ dataset' (Apache 2.0) — OK for fine-tuning  RAG source 'Product documentation' — no derivative work risk (retrieval only)OVERALL STATUS: REVIEW NEEDED```---

## Step 5: Unity Catalog Lineage for License TrackingUnity Catalog tracks **data lineage** — where data came from and where it goes. This iscritical for license compliance because you need to prove the chain of custody forany data that flows into your GenAI application.**UI →** Left nav → **Catalog** → Navigate to a table → **Lineage** tabThe Lineage tab shows:- **Upstream** — where the data came from (source tables, notebooks)- **Downstream** — what depends on this data (models, dashboards)This creates an automatic audit trail for license compliance.

In [ ]:
# Simulating lineage tracking with a simple data structure
lineage_graph = {
    "workspace.genai_labs.raw_documents": {
        "type": "Delta Table",
        "license": "Internal — proprietary",
        "upstream": ["Azure Blob Storage (internal docs)"],
        "downstream": ["workspace.genai_labs.chunks"]
    },
    "workspace.genai_labs.chunks": {
        "type": "Delta Table",
        "license": "Inherited from raw_documents",
        "upstream": ["workspace.genai_labs.raw_documents", "Notebook: 02_chunking_strategies"],
        "downstream": ["workspace.genai_labs.chunk_embeddings", "Vector Search Index"]
    },
    "workspace.genai_labs.chunk_embeddings": {
        "type": "Delta Table",
        "license": "Inherited + embedding model license (MIT - bge-large-en)",
        "upstream": ["workspace.genai_labs.chunks", "databricks-bge-large-en (MIT)"],
        "downstream": ["Vector Search Index", "workspace.genai_labs.rag_chain_v1"]
    },
    "workspace.genai_labs.rag_chain_v1": {
        "type": "MLflow Model (UC)",
        "license": "Composite: Internal data + Llama Community + MIT embeddings",
        "upstream": [
            "workspace.genai_labs.chunk_embeddings",
            "databricks-meta-llama-3-3-70b-instruct (Llama Community)"
        ],
        "downstream": ["Serving Endpoint: rag-bot-prod"]
    }
}

print("DATA LINEAGE — LICENSE TRACE")
print("=" * 70)
for asset, info in lineage_graph.items():
    print(f"\n  {asset}")
    print(f"  Type: {info['type']}")
    print(f"  License: {info['license']}")
    print(f"  Upstream:  {' <- '.join(info['upstream'])}")
    print(f"  Downstream: {' -> '.join(info['downstream'])}")

print("\n" + "=" * 70)
print("EXAM TIP: Unity Catalog lineage lets you trace license obligations")
print("from source data all the way through to the serving endpoint.")
print("This is how you answer 'prove your model is compliant'.")

**Expected output:**```DATA LINEAGE — LICENSE TRACE======================================================================  workspace.genai_labs.raw_documents  Type: Delta Table  License: Internal — proprietary  Upstream:  Azure Blob Storage (internal docs)  Downstream: workspace.genai_labs.chunks  workspace.genai_labs.chunks  ...  workspace.genai_labs.rag_chain_v1  Type: MLflow Model (UC)  License: Composite: Internal data + Llama Community + MIT embeddings  Upstream:  workspace.genai_labs.chunk_embeddings <- databricks-meta-llama-3-3-70b-instruct (Llama Community)  Downstream: Serving Endpoint: rag-bot-prod```---

## Exam Tips and Traps| # | Tip | Why It Matters ||---|-----|---------------|| 1 | **Apache 2.0 includes a patent grant; MIT does not** | Enterprise preference question — Apache 2.0 is safer || 2 | **GPL is copyleft — derivative works must be GPL** | If you fine-tune on GPL data, your model is GPL || 3 | **Fine-tuning = derivative work; RAG = not derivative** | Most important distinction for the exam || 4 | **Llama has a 700M monthly user threshold** | Below 700M = free commercial use; above = need Meta approval || 5 | **Foundation Model APIs handle licensing for you** | Databricks pre-vets these; Marketplace models are YOUR responsibility || 6 | **Unity Catalog lineage traces license obligations** | Used for compliance audits — shows data provenance end-to-end |---

## Key Takeaways### Core Concepts- Five license families: **Apache 2.0** (permissive + patent), **MIT** (permissive), **GPL** (copyleft), **Llama Community** (semi-permissive), **Proprietary** (restricted)- **Fine-tuning creates derivative works** that inherit the training data's license- **RAG does NOT create derivative works** — data is retrieved at runtime, not baked into weights### Databricks-Specific- **Foundation Model APIs** — license pre-vetted by Databricks- **Marketplace models** — check the model card for license terms before deploying- **Unity Catalog lineage** — traces data provenance from source to serving endpoint### Exam Essentials- Apache 2.0 is preferred over MIT for enterprise (patent grant)- GPL on training data = your fine-tuned model must also be GPL- RAG vs fine-tuning is the key derivative works distinction- Llama Community License has a 700M MAU threshold for special terms---**Next Lab →** [Lab 06-01: MLflow RAG Evaluation](../06_evaluation_monitoring/01_mlflow_rag_evaluation.ipynb)